# 1. Baseline cross-encoder

Longformer cross-encoder that classifies whether two abstracts share a KEGG pathway. GPU runtime.
Results go to `BASE/result_crossencoder/run-*`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, json, os

REPO = 'https://github.com/errory230/pathway-supervised-embedder.git'
CODE = '/content/pathway-supervised-embedder'
BASE = '/content/drive/MyDrive/kegg-bge'
DATA = '/content/data/splits'

if os.path.isdir(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO} {CODE}
S = f'{CODE}/scripts'
!mkdir -p /content/data {BASE}
!cp -r {BASE}/data/splits /content/data/
def latest(d): return sorted(glob.glob(f'{d}/run-*'))[-1]

## Training
Best epoch chosen on validation AUROC, early stopping after 2 epochs without improvement; the test set is evaluated once with the best checkpoint.

In [ ]:
OUT = f'{BASE}/result_crossencoder'
!python {S}/train_crossencoder.py --data-dir {DATA} --out-dir {OUT} --epochs 5 --bf16

In [ ]:
import pandas as pd
RUN = latest(OUT); print(RUN)
display(pd.read_json(f'{RUN}/epoch_metrics.jsonl', lines=True).set_index('epoch').round(4).T)
if os.path.exists(f'{RUN}/test_metrics.json'):
    print(json.dumps(json.load(open(f'{RUN}/test_metrics.json')), indent=2))

## Resume after a disconnect
Run the setup cell again, then this cell. Training restarts from the last finished epoch.

In [ ]:
!python {S}/train_crossencoder.py --resume {latest(OUT)} --data-dir {DATA} --epochs 5